# Amazon Nova Forge SDK — Data Preparation Quick Start

This notebook walks through a complete data preparation pipeline:
**load → filter → transform → validate → save**.

We start with intentionally messy raw data (duplicates, URL-heavy records, clean Q/A pairs),
then clean it up step by step into training-ready format.

## What You'll Learn

1. Creating the IAM role required for Glue-based filtering
2. Loading raw JSONL data from S3
3. Filtering out URL-heavy records and exact duplicates (runs on AWS Glue)
4. Transforming filtered data into Converse format for SFT training
5. Validating the transformed dataset against training requirements
6. Saving the final dataset to S3

## Prerequisites

- AWS credentials configured with IAM permissions for running Glue jobs, reading/write from S3, and creating execution IAM roles
- An S3 bucket containing your raw training data (or we'll upload sample data below)

## Step 1: Install and Import

In [ ]:
!cd ../ && pip install .

In [ ]:
from amzn_nova_forge import (
    FilterMethod,
    JSONLDatasetLoader,
    Model,
    Platform,
    TrainingMethod,
    TransformMethod,
    ValidateMethod,
)
from amzn_nova_forge.manager import GlueRuntimeManager

print("SDK imported successfully")

## Step 2: Configuration

Set your S3 bucket and region. The bucket must already exist.

In [ ]:
import boto3

# ============================================================
# TODO: Replace these with your own values
S3_BUCKET = "<your-bucket-here>"
REGION = "us-east-1"
# ============================================================

INPUT_KEY = "nova-forge-sdk/data-prep-quickstart/raw_data.jsonl"
OUTPUT_KEY = "nova-forge-sdk/data-prep-quickstart/prepared_data.jsonl"

INPUT_PATH = f"s3://{S3_BUCKET}/{INPUT_KEY}"
OUTPUT_PATH = f"s3://{S3_BUCKET}/{OUTPUT_KEY}"

session = boto3.Session(region_name=REGION)
account_id = session.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}, Region: {REGION}")
print(f"Input:   {INPUT_PATH}")
print(f"Output:  {OUTPUT_PATH}")

## Step 3: Create Glue Execution Role (One-Time Setup)

Text filters (`DEFAULT_TEXT_FILTER`, `EXACT_DEDUP`, `FUZZY_DEDUP`) run on AWS Glue.
Glue needs an IAM role it can assume. This cell creates the role with the
policies defined in `src/amzn_nova_forge/iam/glue_policies.json`.

**Skip the second cell if you already have a `GlueDataPrepExecutionRole` in your account.**

In [ ]:
GLUE_ROLE_NAME = "NovaForgeSDK-Demo-GlueDataPrepExecutionRole"

In [ ]:
import json

iam = session.client("iam")

# Trust policy: allow Glue to assume this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "glue.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

# Create the role (skip if it already exists)
try:
    iam.create_role(
        RoleName=GLUE_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for Nova Forge SDK Glue data prep jobs",
    )
    print(f"Created role: {GLUE_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"Role already exists: {GLUE_ROLE_NAME}")

# S3 policy: read/write access to the data bucket and the auto-created artifact bucket
artifact_bucket = f"sagemaker-forge-dataprep-{account_id}-{REGION}"
s3_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{S3_BUCKET}",
                f"arn:aws:s3:::{S3_BUCKET}/*",
                f"arn:aws:s3:::{artifact_bucket}",
                f"arn:aws:s3:::{artifact_bucket}/*",
            ],
        }
    ],
}

# CloudWatch Logs policy: Glue writes job logs here
logs_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "arn:aws:logs:*:*:log-group:/aws-glue/*",
        }
    ],
}

# Attach inline policies
for name, doc in [("GlueDataPrepS3Access", s3_policy), ("GlueDataPrepLogs", logs_policy)]:
    iam.put_role_policy(RoleName=GLUE_ROLE_NAME, PolicyName=name, PolicyDocument=json.dumps(doc))
    print(f"Attached inline policy: {name}")

# Attach AWS managed Glue service role policy
iam.attach_role_policy(
    RoleName=GLUE_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole",
)
print(f"Attached managed policy: AWSGlueServiceRole")
print(f"\nGlue role ready: {GLUE_ROLE_NAME}")

## Step 4: Upload Sample Data

We create intentionally messy data to demonstrate the value of each pipeline step:
- **Clean records** — good Q/A pairs that should survive the pipeline
- **URL-heavy records** — text dominated by URLs (simulating web scrape boilerplate)
- **Exact duplicates** — repeated records that dedup should catch

**If you already have data in S3**, skip this cell and point `INPUT_PATH` at your file.

In [ ]:
clean_records = [
    {
        "question": "What is machine learning?",
        "answer": "Machine learning is a subset of AI that enables systems to learn from data without being explicitly programmed.",
    },
    {
        "question": "Explain cloud computing.",
        "answer": "Cloud computing delivers computing resources on demand over the internet, including servers, storage, and databases.",
    },
    {
        "question": "What is Python used for?",
        "answer": "Python is used for web development, data analysis, AI, scientific computing, and automation.",
    },
    {
        "question": "What is an API?",
        "answer": "An API is a set of protocols that allows different software applications to communicate with each other.",
    },
    {
        "question": "Describe containerization.",
        "answer": "Containerization packages an application with its dependencies into a standardized unit for consistent deployment across environments.",
    },
]

# Records with excessive URL content (should be removed by DEFAULT_TEXT_FILTER)
url_heavy_records = [
    {
        "question": "Links?",
        "answer": "Visit https://example.com/page1 https://example.com/page2 https://example.com/page3 https://example.com/page4 https://example.com/page5 https://example.com/page6 https://example.com/page7 https://example.com/page8",
    },
    {
        "question": "More links",
        "answer": "See https://a.com/1 https://b.com/2 https://c.com/3 https://d.com/4 https://e.com/5 https://f.com/6 https://g.com/7 https://h.com/8 https://i.com/9 https://j.com/10",
    },
]

# Build dataset: clean + URL-heavy + exact duplicates of the clean set
raw_data = clean_records + url_heavy_records + clean_records

s3 = session.client("s3")
jsonl_body = "\n".join(json.dumps(r) for r in raw_data) + "\n"
s3.put_object(Bucket=S3_BUCKET, Key=INPUT_KEY, Body=jsonl_body.encode())

print(f"Uploaded {len(raw_data)} records to {INPUT_PATH}")
print(f"  - {len(clean_records)} clean records")
print(f"  - {len(url_heavy_records)} URL-heavy records")
print(f"  - {len(clean_records)} exact duplicates")

## Step 5: Create Runtime Manager

The `GlueRuntimeManager` handles job submission to AWS Glue.
We point it at our S3 bucket for storing Glue scripts and wheels.

In [ ]:
glue_manager = GlueRuntimeManager(
    glue_role_name=GLUE_ROLE_NAME,
    num_workers=2,
    region=REGION,
)
print(f"GlueRuntimeManager ready (role={GLUE_ROLE_NAME}, workers=2)")

## Step 6: Load and Preview Raw Data

In [ ]:
loader = JSONLDatasetLoader()
loader.load(INPUT_PATH)

loader.show(n=5)

## Step 7: Filter — Remove URL-Heavy Records

`DEFAULT_TEXT_FILTER` removes records where URLs dominate the text content.
This runs on AWS Glue via the runtime manager we created above.

In [ ]:
loader.filter(
    method=FilterMethod.DEFAULT_TEXT_FILTER,
    text_field="answer",
    runtime_manager=glue_manager,
)

## Step 8: Filter — Remove Exact Duplicates

`EXACT_DEDUP` removes records that are byte-for-byte identical.
This also runs on AWS Glue.

In [ ]:
loader.filter(
    method=FilterMethod.EXACT_DEDUP,
    text_field="answer",
    runtime_manager=glue_manager,
)

## Step 9: Transform — Convert to Converse Format

Transforms the raw Q/A fields into the Converse schema required for SFT training.

In [ ]:
loader.transform(
    method=TransformMethod.SCHEMA,
    training_method=TrainingMethod.SFT_LORA,
    model=Model.NOVA_LITE_2,
    column_mappings={"question": "question", "answer": "answer"},
)

## Step 10: Show the Filtered and Transformed Data

Calling a terminal operation like "show" causes the chain of queued steps to execute.


In [ ]:
loader.show(n=3)

## Step 11: Validate — Check Training Requirements

Validates that every record conforms to the SFT schema
(correct fields, role alternation, no forbidden keywords, etc.).
Raises an error if any records are invalid.

In [ ]:
loader.validate(
    method=ValidateMethod.INVALID_RECORDS,
    training_method=TrainingMethod.SFT_LORA,
    model=Model.NOVA_LITE_2,
)
print("Validation passed!")

## Step 12: Save — Write Training-Ready Data to S3

In [ ]:
loader.save(OUTPUT_PATH)

## Summary

We started with 12 messy records and ended up with 5 clean, deduplicated, schema-valid training samples:

| Step | Records | What happened |
|------|---------|---------------|
| Load | 12 | Raw data with duplicates and URL spam |
| Text Filter | 10 | Removed 2 URL-heavy records |
| Exact Dedup | 5 | Removed 5 duplicate records |
| Transform | 5 | Converted to Converse format |
| Validate | 5 | All records pass SFT schema checks |
| Save | 5 | Written to S3 as training-ready JSONL |

## Next Steps

- See `docs/data_prep.md` for the full data preparation guide (fuzzy dedup, splitting, multimodal, etc.)
- Use the saved data with `NovaModelCustomizer` for training — see `samples/nova_quickstart.ipynb`